# Дообучение (continual fine-tuning) mT5-small на config-conditioned датасете

Это ПРОДОЛЖЕНИЕ обучения уже готовой модели
`ismailoviskandar02/uzbek-text-simplifier`, подпапка `model2` на HF Hub
(не `model` -- та подпапка со старым чекпоинтом, `model2` -- то, что нужно
дообучать сейчас). Грузим с Hub через `subfolder`, не с локального диска --
так не зависим от того, жива ли ещё файловая система Colab-рантайма между
сессиями.

Новые данные: `combined_simplified_clean_train_ready.csv`, колонки
`prefix` / `input_text` / `target_text`. В датасете 9 разных
prefix-вариантов (aggressiveness/max_length/drop_* и их комбинации),
уже отфильтрованных от severely-mismatched пар (2587 строк, ratio-фильтр
по `target*0.5` и `>1.3`).

Задача этого дообучения -- не "научить упрощать" (это модель уже умеет),
а научить её МОДУЛИРОВАТЬ поведение в зависимости от текста в prefix.
Из этого следует главное отличие от прошлого ноутбука: input в модель
собирается как `prefix + input_text` (prefix уже содержит финальное
`": "`), а не `PREFIX_CONST + text`.

**Сохранённые фиксы из прошлого прогона (не трогать):**
- `fp16=False` -- баг переполнения/NaN loss у mT5 в fp16.
- `low_cpu_mem_usage=False` при загрузке -- иначе tied-weights
  (`encoder/decoder.embed_tokens.weight`) не подгружаются из чекпоинта.
- `processing_class=tokenizer` (не `tokenizer=`) в `Seq2SeqTrainer`.
- Ручная перезагрузка лучшего чекпоинта после `trainer.train()` --
  `load_best_model_at_end=True` триггерит тот же tied-weights баг повторно.

**Runtime → T4 GPU**

In [ ]:
!pip install -q transformers datasets evaluate sentencepiece accelerate rouge_score sacrebleu

## 1. Загрузка config-conditioned датасета

In [ ]:
from google.colab import files
import os

needed = ['combined_simplified_clean_train_ready.csv']
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print('Загрузи файлы:', missing)
    uploaded = files.upload()

# Модель тянется напрямую с HF Hub в следующей секции (subfolder='model2') --
# локально ничего загружать не нужно.

In [ ]:
import pandas as pd

df = pd.read_csv('combined_simplified_clean_train_ready.csv')
df = df.dropna(subset=['prefix', 'input_text', 'target_text'])
df = df[df['input_text'].str.strip() != '']
df = df[df['target_text'].str.strip() != '']

# дедуп по (prefix, input_text) -- не просто по input_text, т.к. один и
# тот же исходный текст физически не повторяется у нас (каждая строка
# получила ровно один конфиг через config_for_row), но дублирующиеся
# чанки внутри длинных документов теоретически возможны после чанкинга
df = df.drop_duplicates(subset=['prefix', 'input_text'])

print('Всего чистых примеров:', len(df))
print()
print('Распределение по prefix (конфигам):')
print(df['prefix'].value_counts())
df.head(2)

### Проверка баланса по конфигам

Если какой-то prefix заметно отстаёт по количеству примеров от
остальных (см. предыдущий разбор: ориентир ~250-300 как нижний порог,
~500+ комфортно) -- модель хуже свяжет именно этот конфиг с нужным
поведением. Ячейка ниже просто делает это видимым перед тем, как
тратить GPU-время на обучение.

In [ ]:
counts = df['prefix'].value_counts()
min_count = counts.min()
print(f'Минимум по конфигу: {min_count} ({counts.idxmin()!r})')
if min_count < 150:
    print('⚠️  Меньше 150 примеров на этот конфиг -- модель, скорее всего, '
          'не выучит его надёжно. Стоит либо догнать этот конфиг данными, '
          'либо принять, что он останется слабым.')
elif min_count < 300:
    print('⚠️  Ниже комфортного порога (~300), но выше минимального (~150) -- '
          'рабочий, но не идеальный случай.')
else:
    print('✅ Все конфиги выше комфортного порога.')

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# stratify по prefix, чтобы в validation тоже попали примеры каждого
# конфига, а не только самых частых -- иначе метрики на eval будут
# смещены к доминирующим конфигам
train_df, val_df = train_test_split(
    df, test_size=0.05, random_state=42, stratify=df['prefix']
)

ds = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
})
print(ds)

## 2. Загрузка ГОТОВОЙ модели с HF Hub (subfolder `model2`)

Модель лежит в репозитории `ismailoviskandar02/uzbek-text-simplifier`,
в подпапке `model2` (см. Files and versions на странице репо -- рядом
есть ещё `model/`, это другой, более старый чекпоинт, не путать).

`low_cpu_mem_usage=False` оставляем -- та же причина, что и раньше
(tied-weights баг при загрузке через meta-device).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "ismailoviskandar02/uzbek-text-simplifier"
SUBFOLDER = "model2"

# Данные содержат чанки от коротких (10-100 слов) до довольно длинных
# (high-ratio конфиги режутся на ~1500 симв., но balanced/aggressive
# чанки могут доходить до 6000 симв. по построению пайплайна) --
# поэтому длину контекста НЕ уменьшаем относительно базовой модели, а
# скорее увеличиваем, чтобы не обрезать длинные чанки на середине фразы.
MAX_INPUT_LEN = 384
MAX_TARGET_LEN = 384

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, subfolder=SUBFOLDER)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, subfolder=SUBFOLDER, low_cpu_mem_usage=False
)
model.config.tie_word_embeddings = False  # чекпоинт хранит shared/lm_head отдельно

# sanity-check: эмбеддинги должны быть ОБУЧЕННЫМИ, не случайными
emb = model.get_input_embeddings().weight
print("embedding mean/std:", emb.mean().item(), emb.std().item())
print("(std должен быть в районе 10+ -- если ~0.02-0.05, эмбеддинги случайные, что-то не так)")

In [ ]:
def preprocess(batch):
    # prefix уже содержит финальное ": " (например
    # 'simplify [aggressiveness=conservative, max_length=0.8]: ') --
    # просто конкатенируем с исходным текстом, никакой отдельной
    # константы PREFIX здесь больше нет.
    inputs = [p + t for p, t in zip(batch['prefix'], batch['input_text'])]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch['target_text'], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)

## 3. Метрики (без изменений)

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load('rouge')
chrf = evaluate.load('chrf')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    chrf_result = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    result['chrf'] = chrf_result['score']
    return {k: round(v, 4) if isinstance(v, float) else v for k, v in result.items()}

## 4. Дообучение

- `learning_rate=1e-4` -- продолжаем обучение уже сошедшейся модели, не с нуля.
- `num_train_epochs=4` -- conditioning на 9 конфигах при ~2600 примерах
  (в среднем ~290 на конфиг) сходится медленнее, чем просто "упрощай";
  если по логам chrF всё ещё растёт на последней эпохе -- увеличивай и
  перезапускай, `load_best_model_at_end` всё равно вернёт лучший чекпоинт.
- `per_device_train_batch_size=4` (было 8) -- часть чанков теперь длиннее
  (до `MAX_INPUT_LEN=384` вместо 192), больше памяти на пример;
  `gradient_accumulation_steps=8`, чтобы эффективный batch size остался
  тем же (32), что и в прошлом прогоне.
- `fp16=False` -- ОБЯЗАТЕЛЬНО для mT5.

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import torch

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir='mt5_uz_simplify_config_conditioned',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.02,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='chrf',
    fp16=False,  # КРИТИЧНО для mT5 -- вызывает NaN loss при включении
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
train_result = trainer.train()

# см. объяснение в шапке ноутбука: load_best_model_at_end триггерит
# tied-weights баг повторно при внутренней перезагрузке -- чиним вручную.
best_ckpt = trainer.state.best_model_checkpoint
print("Лучший чекпоинт:", best_ckpt)

model = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt, low_cpu_mem_usage=False)
model.config.tie_word_embeddings = False
model.to(trainer.args.device)

emb = model.get_input_embeddings().weight
print("embedding mean/std (после перезагрузки):", emb.mean().item(), emb.std().item())
print("(std должен остаться в районе 10+ -- если упал к ~0.02-0.05, баг снова сработал)")

trainer.model = model

## 5. Проверка conditioning'а: один и тот же текст, разные конфиги

В отличие от прошлого ноутбука (там сравнивали ДО/ПОСЛЕ на одном
prefix), здесь главная проверка другая: гоняем ОДИН и тот же исходный
текст через несколько разных prefix из validation-набора конфигов и
смотрим, что output реально отличается по длине/содержанию в
предсказанную сторону, а не одинаков независимо от prefix (что было бы
признаком того, что conditioning не выучился).

In [ ]:
def simplify_with_prefix(prefix, text, max_new_tokens=300):
    inputs = tokenizer(prefix + text, return_tensors='pt', truncation=True,
                        max_length=MAX_INPUT_LEN).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

sample_text = val_df['input_text'].iloc[0]
all_prefixes = sorted(df['prefix'].unique())

print('SOURCE TEXT:', sample_text[:200], '...\n')
for p in all_prefixes:
    out = simplify_with_prefix(p, sample_text)
    print(f'[{p}]')
    print(f'  -> ({len(out)} chars) {out[:200]}')
    print()

## 6. Сохранение и скачивание

In [ ]:
trainer.save_model('mt5_uz_simplify_config_conditioned_final')
tokenizer.save_pretrained('mt5_uz_simplify_config_conditioned_final')

!zip -r mt5_uz_simplify_config_conditioned_final.zip mt5_uz_simplify_config_conditioned_final
from google.colab import files as colab_files
colab_files.download('mt5_uz_simplify_config_conditioned_final.zip')

## 7. Обновление модели на Hugging Face Hub

Пушим в тот же репозиторий, в ту же подпапку `model2` (не `model` --
та подпапка не трогается этим прогоном). Это создаст новую версию
(коммит) поверх текущего содержимого `model2/` -- откатить можно в
истории коммитов репозитория.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub(
#     'ismailoviskandar02/uzbek-text-simplifier',
#     commit_message='Continue fine-tuning for prefix/config conditioning (aggressiveness, max_length, drop_*)',
# )
# # push_to_hub кладёт файлы в корень репо, а не в подпапку -- если нужно
# # именно в model2/, залей вручную через huggingface_hub.upload_folder(
# #     folder_path='mt5_uz_simplify_config_conditioned_final',
# #     repo_id='ismailoviskandar02/uzbek-text-simplifier',
# #     path_in_repo='model2',
# # )